# Forecast

In this notebook, we load a Random Forest trained with

```bash
pyrocast-train-rf --config configs/random_forest.yaml
```

(`split.scheme: holdout`) and make predictions over its test set. Paths in the config are relative to the repository root, so the notebook moves there first.

In [ ]:
import os
from pathlib import Path

import joblib

from pyrocast.config import RFExperimentConfig, load_config
from pyrocast.utils import metrics
from pyrocast.utils.data.cross_validation import split_by_fire
from pyrocast.utils.data.dataload import get_sample_index
from pyrocast.utils.data.features import (
    get_feature_table,
    input_variables,
    sample_features,
)

# run from the repository root, wherever the notebook was started
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)

## Load data and model

In [ ]:
cfg = load_config("configs/random_forest.yaml", RFExperimentConfig)
index = get_sample_index(cfg.data, cfg.mode)
rf_model = joblib.load(cfg.run_dir / "model.joblib")

## Prepare data

The split is grouped by wildfire and seeded by the config, so this recovers the test set of a `holdout` run. The features come from the cache at `data.feature_cache`.

In [ ]:
_, test = split_by_fire(index, cfg.split)
variables = input_variables(cfg.inputs, cfg.era5_variables)
xtest, _ = sample_features(test, get_feature_table(cfg.data, test), variables)
ytest = test.label.to_numpy()
print(f"{len(test)} test samples, {ytest.sum()} positive, {xtest.shape[1]} features")

## Make predictions

In [ ]:
ypred = rf_model.predict_proba(xtest)[:, 1]
auc = metrics.auc(ytest, ypred)
print(auc)